In [1]:
!pip install torch torchvision

In [2]:
import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Đã cố định Random Seed = 42 cho toàn bộ hệ thống.")

Đã cố định Random Seed = 42 cho toàn bộ hệ thống.


In [3]:
#Tạo tệp ánh xạ nhãn chung (class_to_idx.json)
import os
import json
import pandas as pd
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebook" or os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# 1. Tạo class_to_idx.json chung
train_df = pd.read_csv("data/splits/train.csv")
classes = sorted(train_df['class_name'].unique().tolist())
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

with open(models_dir / "class_to_idx.json", "w", encoding="utf-8") as f:
    json.dump(class_to_idx, f, ensure_ascii=False, indent=4)

print(f"Đã lưu ánh xạ nhãn chung tại models/class_to_idx.json: {class_to_idx}")

Đã lưu ánh xạ nhãn chung tại models/class_to_idx.json: {'Bacterial Gill Diseases': 0, 'Bacterial Red Diseases': 1, 'Epizootic Ulcerative Syndrome (EUS)': 2, 'Healthy Fish': 3}


In [4]:
#Định nghĩa FishDataset dùng chung class_to_idx.json và tính Class Weights
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class FishDataset(Dataset):
    def __init__(self, csv_file, class_to_idx_path, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        
        with open(class_to_idx_path, "r", encoding="utf-8") as f:
            self.class_to_idx = json.load(f)
            
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        label_name = row['class_name']
        
        image = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[label_name]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Tính class weights cho CrossEntropyLoss (CHỈ TÍNH TRÊN TẬP TRAIN)
train_labels = [class_to_idx[name] for name in train_df['class_name']]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"Class Weights tính trên tập Train: {class_weights}")

Class Weights tính trên tập Train: [2.01635514 0.87348178 1.78305785 0.5560567 ]


In [5]:
import json
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

# Đọc file ánh xạ nhãn class_to_idx.json đã lưu ở bước trước
models_dir = Path("models")
json_path = models_dir / "class_to_idx.json"

with open(json_path, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

# Định nghĩa các phép biến đổi ảnh (Pre-processing & Augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Khởi tạo các Dataset (Đảm bảo lớp FishDataset đã được định nghĩa ở cell phía trên)
train_dataset = FishDataset("data/splits/train.csv", json_path, transform=train_transform)
val_dataset = FishDataset("data/splits/val.csv", json_path, transform=val_transform)
test_dataset = FishDataset("data/splits/test.csv", json_path, transform=val_transform)

# Khởi tạo DataLoader cho 3 tập Train, Validation, Test
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print("Đã khởi tạo xong DataLoader cho Train, Val, Test thành công!")

Đã khởi tạo xong DataLoader cho Train, Val, Test thành công!


In [7]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from torchvision import models

# Thiết lập thiết bị an toàn
device = DEVICE if 'DEVICE' in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- HÀM CHUNG KHỞI TẠO MÔ HÌNH ---
def build_model(model_name, num_classes=4):
    if model_name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == "mobilenet_v3":
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f"Model không hỗ trợ: {model_name}")
    return m.to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

# --- LỚP EARLY STOPPING CHỐNG OVERFITTING ---
class EarlyStopping:
    def __init__(self, patience=4, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_metric):
        score = val_metric
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0

# --- DANH SÁCH 4 MÔ HÌNH CẦN CHẠY ---
model_names = ["resnet18", "resnet50", "efficientnet_b0", "mobilenet_v3"]
comparison_results = []
num_epochs = 15  # Đặt số epoch tối đa, EarlyStopping sẽ tự động dừng sớm

class_weights_tensor = class_weights_tensor.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

for m_name in model_names:
    print(f"\n==============================================")
    print(f"BẮT ĐẦU HUẤN LUYỆN & ĐÁNH GIÁ MÔ HÌNH: {m_name.upper()}")
    print(f"==============================================")
    
    start_time = time.time()
    model = build_model(m_name, num_classes=len(class_to_idx))
    n_params = count_parameters(model)
    
    # Dùng learning rate nhỏ hơn (1e-4) để fine-tuning ổn định
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)
    
    # Khởi tạo Scheduler (đã bỏ verbose) và Early Stopping
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    early_stopping = EarlyStopping(patience=4, min_delta=0.001)
    
    best_val_f1 = 0.0
    best_model_path = models_dir / f"{m_name}_best.pth"
    
    for epoch in range(num_epochs):
        # --- TRAINING ---
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            
        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                
                _, predicted = outputs.max(1)
                val_preds.extend(predicted.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
                
        val_loss = val_loss / len(val_loader.dataset)
        epoch_val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        
        # Giảm learning rate tự động khi val_loss chững lại
        scheduler.step(val_loss)
        
        # Kiểm tra Early Stopping theo Val Macro-F1
        early_stopping(epoch_val_f1)
        
        # Lưu checkpoint theo Best Validation Macro-F1
        if epoch_val_f1 > best_val_f1:
            best_val_f1 = epoch_val_f1
            torch.save(model.state_dict(), best_model_path)
            
        print(f"Epoch [{epoch+1}/{num_epochs}] | Val Loss: {val_loss:.4f} | Val Macro-F1: {epoch_val_f1:.4f} (Best Val F1: {best_val_f1:.4f})")
        
        if early_stopping.early_stop:
            print(f"Early stopping kích hoạt tại epoch {epoch+1} cho mô hình {m_name} để tránh overfitting!")
            break
            
    training_time = time.time() - start_time
    
    # --- ĐÁNH GIÁ TRÊN TẬP TEST (DÙNG MODEL TỐT NHẤT) ---
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    
    test_preds, test_targets = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            test_preds.extend(predicted.cpu().numpy())
            test_targets.extend(labels.numpy())
            
    test_acc = accuracy_score(test_targets, test_preds)
    test_precision, test_recall, test_macro_f1, _ = precision_recall_fscore_support(test_targets, test_preds, average='macro', zero_division=0)
    
    # Lưu vào danh sách tổng hợp bảng so sánh
    comparison_results.append({
        "Model": m_name,
        "Best Val Macro-F1": round(best_val_f1, 4),
        "Test Accuracy": round(test_acc, 4),
        "Precision": round(test_precision, 4),
        "Recall": round(test_recall, 4),
        "Macro-F1": round(test_macro_f1, 4),
        "Parameters (M)": round(n_params / 1e6, 2),
        "Training Time (s)": round(training_time, 2)
    })

# --- XUẤT FILE TỔNG HỢP MODEL_COMPARISON.CSV ---
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)
df_comparison = pd.DataFrame(comparison_results)
df_comparison.to_csv(reports_dir / "model_comparison.csv", index=False)

print("\n==============================================")
print("BẢNG TỔNG HỢP SO SÁNH 4 MÔ HÌNH (reports/model_comparison.csv):")
print("==============================================")
print(df_comparison.to_string(index=False))


BẮT ĐẦU HUẤN LUYỆN & ĐÁNH GIÁ MÔ HÌNH: RESNET18
Epoch [1/15] | Val Loss: 0.5579 | Val Macro-F1: 0.7679 (Best Val F1: 0.7679)
Epoch [2/15] | Val Loss: 0.9232 | Val Macro-F1: 0.7102 (Best Val F1: 0.7679)
Epoch [3/15] | Val Loss: 1.0129 | Val Macro-F1: 0.7295 (Best Val F1: 0.7679)
Epoch [4/15] | Val Loss: 0.8106 | Val Macro-F1: 0.7568 (Best Val F1: 0.7679)
Epoch [5/15] | Val Loss: 0.8778 | Val Macro-F1: 0.7312 (Best Val F1: 0.7679)
Early stopping kích hoạt tại epoch 5 cho mô hình resnet18 để tránh overfitting!

BẮT ĐẦU HUẤN LUYỆN & ĐÁNH GIÁ MÔ HÌNH: RESNET50
Epoch [1/15] | Val Loss: 0.7602 | Val Macro-F1: 0.7261 (Best Val F1: 0.7261)
Epoch [2/15] | Val Loss: 0.6790 | Val Macro-F1: 0.7306 (Best Val F1: 0.7306)
Epoch [3/15] | Val Loss: 0.9791 | Val Macro-F1: 0.7213 (Best Val F1: 0.7306)
Epoch [4/15] | Val Loss: 0.7130 | Val Macro-F1: 0.7622 (Best Val F1: 0.7622)
Epoch [5/15] | Val Loss: 0.6980 | Val Macro-F1: 0.7682 (Best Val F1: 0.7682)
Epoch [6/15] | Val Loss: 0.8300 | Val Macro-F1: 0.77

100%|██████████| 9.83M/9.83M [00:00<00:00, 10.9MB/s]


Epoch [1/15] | Val Loss: 0.8894 | Val Macro-F1: 0.3946 (Best Val F1: 0.3946)
Epoch [2/15] | Val Loss: 0.8043 | Val Macro-F1: 0.5724 (Best Val F1: 0.5724)
Epoch [3/15] | Val Loss: 0.8433 | Val Macro-F1: 0.6215 (Best Val F1: 0.6215)
Epoch [4/15] | Val Loss: 0.7721 | Val Macro-F1: 0.7389 (Best Val F1: 0.7389)
Epoch [5/15] | Val Loss: 1.0728 | Val Macro-F1: 0.7174 (Best Val F1: 0.7389)
Epoch [6/15] | Val Loss: 1.1622 | Val Macro-F1: 0.6946 (Best Val F1: 0.7389)
Epoch [7/15] | Val Loss: 1.2110 | Val Macro-F1: 0.6946 (Best Val F1: 0.7389)
Epoch [8/15] | Val Loss: 1.3334 | Val Macro-F1: 0.7131 (Best Val F1: 0.7389)
Early stopping kích hoạt tại epoch 8 cho mô hình mobilenet_v3 để tránh overfitting!

BẢNG TỔNG HỢP SO SÁNH 4 MÔ HÌNH (reports/model_comparison.csv):
          Model  Best Val Macro-F1  Test Accuracy  Precision  Recall  Macro-F1  Parameters (M)  Training Time (s)
       resnet18             0.7679         0.8222     0.8471  0.8547    0.8386           11.18            1371.60
       

## PHẦN 6: KHÓA PROTOCOL LỰA CHỌN MÔ HÌNH (MODEL SELECTION PROTOCOL)

### 6.1. Nguyên tắc phương pháp luận khoa học
- **Quy tắc vàng trong Machine Learning**: Tập **Test (225 ảnh - 16 cá thể)** tuyệt đối không được sử dụng để điều chỉnh siêu tham số (hyperparameters), chọn checkpoint hay so sánh để chọn kiến trúc mạng cuối cùng. Việc chọn mô hình chỉ vì Test F1 cao nhất sẽ dẫn đến hiện tượng **rò rỉ dữ liệu (data leakage / test-set overfitting)**.
- **Protocol chuẩn hóa**: Việc lựa chọn mô hình phải được thực hiện hoàn toàn trên tập **Train / Validation** thông qua đánh giá đa hạt giống ngẫu nhiên (Multi-seed Stability Analysis, gồm 5 random seeds: `42, 100, 2024, 777, 999`) để đo lường giá trị kỳ vọng (Mean) và độ ổn định (Standard Deviation $\sigma$) của Macro-F1.
- **Bộ tiêu chí quyết định đa mục tiêu (Multi-criteria Decision Matrix)** gồm:
  1. **Validation Macro-F1 (Mean)**: Hiệu năng nhận diện bệnh kỳ vọng.
  2. **Độ ổn định ($\sigma$)**: Độ lệch chuẩn càng thấp, mô hình càng ít nhạy cảm với khởi tạo ngẫu nhiên.
  3. **Số lượng tham số & Kích thước**: Khả năng triển khai trên thiết bị biên (Edge AI) tại các bè cá.
  4. **Thời gian suy luận (Inference Latency & FPS)**: Khả năng phản hồi thời gian thực.

In [ ]:
# Hiển thị Bảng tổng hợp Giao thức Lựa chọn Mô hình (reports/model_selection_protocol_summary.csv)
import pandas as pd
df_protocol = pd.read_csv('reports/model_selection_protocol_summary.csv')
print('=' * 85)
print('BẢNG TỔNG HỢP PROTOCOL LỰA CHỌN MÔ HÌNH (TRAIN/VAL ONLY - 5 SEEDS):')
print('=' * 85)
print(df_protocol[['Model', 'Val Macro-F1 (Mean ± Std)', 'Parameters (M)', 'Latency (ms/img)', 'Throughput (FPS)']].to_string(index=False))


### 6.2. Quyết định chốt mô hình cuối cùng (Final Model Selection)
Dựa trên bảng đối chiếu đa tiêu chí:
- **EfficientNet-B0** đạt **Validation Macro-F1 trung bình cao nhất (0.7873)** và có **độ ổn định vượt trội tuyệt đối** (độ lệch chuẩn $\sigma = 0.0061$, thấp hơn 10 lần so với ResNet-50 có $\sigma = 0.0645$).
- **Kích thước mô hình tối ưu**: EfficientNet-B0 chỉ có **4.01M tham số** (kích thước ~16 MB), nhẹ hơn gần 6 lần so với ResNet-50 (23.52M tham số, ~94 MB).
- **Tốc độ xử lý**: Đạt độ trễ **22.53 ms/ảnh** (tương đương **44.4 FPS**), nhanh gấp đôi so với ResNet-50 (45.20 ms/ảnh), hoàn toàn đáp ứng yêu cầu chẩn đoán thời gian thực.
- **KẾT LUẬN**: Khóa giao thức và **chính thức chốt EfficientNet-B0 làm mô hình Deep Learning cuối cùng của hệ thống**. Sau bước này, toàn bộ siêu tham số của mô hình được cố định vĩnh viễn và không thay đổi dựa trên tập Test.

## PHẦN 7: ĐÁNH GIÁ CHUYÊN SÂU MÔ HÌNH ĐÃ CHỐT TRÊN TẬP TEST (EFFICIENTNET-B0)

Sau khi đã khóa và chốt mô hình EfficientNet-B0 hoàn toàn dựa trên Train/Val, ta tiến hành đánh giá khách quan duy nhất 1 lần trên tập Test (225 ảnh) gồm:
1. **Ma trận nhầm lẫn (Confusion Matrix)**.
2. **Báo cáo Precision / Recall / F1 từng lớp**.
3. **Phân tích Top-3 Probabilities & Các ảnh phân loại sai điển hình**.
4. **Giải thích trực quan vùng chẩn đoán với Grad-CAM (Explainable AI - XAI)**.

In [ ]:
import pandas as pd
from IPython.display import Image, display

print('=== BÁO CÁO PHÂN LOẠI CHI TIẾT TỪNG LỚP TRÊN TẬP TEST (EFFICIENTNET-B0) ===')
df_metrics = pd.read_csv('reports/final_model_per_class_metrics.csv', index_col=0)
print(df_metrics.round(4))

print('\n=== MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ===')
display(Image(filename='reports/final_model_confusion_matrix.png'))


In [ ]:
print('=== PHÂN TÍCH CÁC CA PHÂN LOẠI SAI ĐIỂN HÌNH (MISCLASSIFIED SAMPLES) ===')
df_mis = pd.read_csv('reports/misclassified_samples_analysis.csv')
print(f'Tổng số mẫu dự đoán sai: {len(df_mis)} / 225 ({len(df_mis)/225*100:.2f}%)\n')
print('Phân bố nhầm lẫn giữa các lớp bệnh:')
print(df_mis.groupby(['true_class', 'predicted_class']).size())

print('\n=== TRỰC QUAN HÓA GIẢI THÍCH CHẨN ĐOÁN GRAD-CAM ===')
display(Image(filename='reports/gradcam_analysis.png'))


## PHẦN 6: KHÓA PROTOCOL LỰA CHỌN MÔ HÌNH (MODEL SELECTION PROTOCOL)

### 6.1. Nguyên tắc phương pháp luận khoa học
- **Quy tắc vàng trong Machine Learning**: Tập **Test (225 ảnh - 16 cá thể)** tuyệt đối không được sử dụng để điều chỉnh siêu tham số (hyperparameters), chọn checkpoint hay so sánh để chọn kiến trúc mạng cuối cùng. Việc chọn mô hình chỉ vì Test F1 cao nhất sẽ dẫn đến hiện tượng **rò rỉ dữ liệu (data leakage / test-set overfitting)**.
- **Protocol chuẩn hóa**: Việc lựa chọn mô hình phải được thực hiện hoàn toàn trên tập **Train / Validation** thông qua đánh giá đa hạt giống ngẫu nhiên (Multi-seed Stability Analysis, gồm 5 random seeds: `42, 100, 2024, 777, 999`) để đo lường giá trị kỳ vọng (Mean) và độ ổn định (Standard Deviation $\sigma$) của Macro-F1.
- **Bộ tiêu chí quyết định đa mục tiêu (Multi-criteria Decision Matrix)** gồm:
  1. **Validation Macro-F1 (Mean)**: Hiệu năng nhận diện bệnh kỳ vọng.
  2. **Độ ổn định ($\sigma$)**: Độ lệch chuẩn càng thấp, mô hình càng ít nhạy cảm với khởi tạo ngẫu nhiên.
  3. **Số lượng tham số & Kích thước**: Khả năng triển khai trên thiết bị biên (Edge AI) tại các bè cá.
  4. **Thời gian suy luận (Inference Latency & FPS)**: Khả năng phản hồi thời gian thực.

In [ ]:
# Hiển thị Bảng tổng hợp Giao thức Lựa chọn Mô hình (reports/model_selection_protocol_summary.csv)
import pandas as pd
df_protocol = pd.read_csv('reports/model_selection_protocol_summary.csv')
print('=' * 85)
print('BẢNG TỔNG HỢP PROTOCOL LỰA CHỌN MÔ HÌNH (TRAIN/VAL ONLY - 5 SEEDS):')
print('=' * 85)
print(df_protocol[['Model', 'Val Macro-F1 (Mean ± Std)', 'Parameters (M)', 'Latency (ms/img)', 'Throughput (FPS)']].to_string(index=False))


### 6.2. Quyết định chốt mô hình cuối cùng (Final Model Selection)
Dựa trên bảng đối chiếu đa tiêu chí:
- **EfficientNet-B0** đạt **Validation Macro-F1 trung bình cao nhất (0.7873)** và có **độ ổn định vượt trội tuyệt đối** (độ lệch chuẩn $\sigma = 0.0061$, thấp hơn 10 lần so với ResNet-50 có $\sigma = 0.0645$).
- **Kích thước mô hình tối ưu**: EfficientNet-B0 chỉ có **4.01M tham số** (kích thước ~16 MB), nhẹ hơn gần 6 lần so với ResNet-50 (23.52M tham số, ~94 MB).
- **Tốc độ xử lý**: Đạt độ trễ **22.53 ms/ảnh** (tương đương **44.4 FPS**), nhanh gấp đôi so với ResNet-50 (45.20 ms/ảnh), hoàn toàn đáp ứng yêu cầu chẩn đoán thời gian thực.
- **KẾT LUẬN**: Khóa giao thức và **chính thức chốt EfficientNet-B0 làm mô hình Deep Learning cuối cùng của hệ thống**. Sau bước này, toàn bộ siêu tham số của mô hình được cố định vĩnh viễn và không thay đổi dựa trên tập Test.

## PHẦN 7: ĐÁNH GIÁ CHUYÊN SÂU MÔ HÌNH ĐÃ CHỐT TRÊN TẬP TEST (EFFICIENTNET-B0)

Sau khi đã khóa và chốt mô hình EfficientNet-B0 hoàn toàn dựa trên Train/Val, ta tiến hành đánh giá khách quan duy nhất 1 lần trên tập Test (225 ảnh) gồm:
1. **Ma trận nhầm lẫn (Confusion Matrix)**.
2. **Báo cáo Precision / Recall / F1 từng lớp**.
3. **Phân tích Top-3 Probabilities & Các ảnh phân loại sai điển hình**.
4. **Giải thích trực quan vùng chẩn đoán với Grad-CAM (Explainable AI - XAI)**.

In [ ]:
import pandas as pd
from IPython.display import Image, display

print('=== BÁO CÁO PHÂN LOẠI CHI TIẾT TỪNG LỚP TRÊN TẬP TEST (EFFICIENTNET-B0) ===')
df_metrics = pd.read_csv('reports/final_model_per_class_metrics.csv', index_col=0)
print(df_metrics.round(4))

print('\n=== MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ===')
display(Image(filename='reports/final_model_confusion_matrix.png'))


In [ ]:
print('=== PHÂN TÍCH CÁC CA PHÂN LOẠI SAI ĐIỂN HÌNH (MISCLASSIFIED SAMPLES) ===')
df_mis = pd.read_csv('reports/misclassified_samples_analysis.csv')
print(f'Tổng số mẫu dự đoán sai: {len(df_mis)} / 225 ({len(df_mis)/225*100:.2f}%)\n')
print('Phân bố nhầm lẫn giữa các lớp bệnh:')
print(df_mis.groupby(['true_class', 'predicted_class']).size())

print('\n=== TRỰC QUAN HÓA GIẢI THÍCH CHẨN ĐOÁN GRAD-CAM ===')
display(Image(filename='reports/gradcam_analysis.png'))
